# Week 3: Informed Search: A* on the 8-Puzzle

A* combines the cost already paid with a heuristic estimate of the remaining path. This notebook implements both heuristics named in the Week 3 slides.

**Source:** `PPTs/Week3.pptx`.


## Program — 8-Puzzle A* with misplaced-tile and Manhattan heuristics

**Theory.** A* ranks frontier states by f(n)=g(n)+h(n), where g is the path cost and h estimates remaining work. The Week 3 slides introduce both the number-of-tiles-out-of-place heuristic and Manhattan distance. Both are admissible for the 8-puzzle when the blank is excluded.

**Objective.** Solve a custom 8-puzzle using either heuristic and compare the search counts.

**Libraries.** Python standard library (`heapq`, `itertools`).

**Algorithm.**
1. Read and validate a solvable board. 2. Put the start state in a priority queue. 3. Pop the state with least g+h. 4. Relax each neighbor with cost g+1. 5. Stop at the goal and reconstruct the path. Repeat for both heuristics.

**Complexity.** Worst case exponential time and space O(b^d); actual work depends strongly on the heuristic.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** Both heuristics find an optimal path; Manhattan distance is at least as informed as misplaced tiles on these puzzle states.


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

import heapq
from itertools import count
GOAL=(1,2,3,4,5,6,7,8,0)
def neighbors(s):
    z=s.index(0); r,c=divmod(z,3)
    for dr,dc in ((-1,0),(1,0),(0,-1),(0,1)):
        nr,nc=r+dr,c+dc
        if 0<=nr<3 and 0<=nc<3:
            j=nr*3+nc; t=list(s); t[z],t[j]=t[j],t[z]; yield tuple(t)
def heuristic(s,kind):
    if kind=='misplaced': return sum(v!=0 and v!=GOAL[i] for i,v in enumerate(s))
    total=0
    for i,v in enumerate(s):
        if v:
            goal_i=v-1; r,c=divmod(i,3); gr,gc=divmod(goal_i,3); total+=abs(r-gr)+abs(c-gc)
    return total
def astar(start,kind):
    serial=count(); frontier=[(heuristic(start,kind),0,next(serial),start)]
    best={start:0}; parent={start:None}; expanded=0
    while frontier:
        _,g,_,state=heapq.heappop(frontier)
        if g!=best.get(state): continue
        if state==GOAL:
            path=[]; cur=state
            while cur is not None: path.append(cur); cur=parent[cur]
            return list(reversed(path)),expanded
        expanded+=1
        for child in neighbors(state):
            ng=g+1
            if ng<best.get(child,float('inf')):
                best[child]=ng; parent[child]=state
                heapq.heappush(frontier,(ng+heuristic(child,kind),ng,next(serial),child))
    return None,expanded
def main():
    try: start=tuple(map(int,input('Enter 9 tiles, 0 for blank: ').split()))
    except ValueError: print('Use nine integers.'); return
    if len(start)!=9 or set(start)!=set(range(9)): print('Tiles must be each integer 0 through 8 exactly once.'); return
    a=[v for v in start if v]; inv=sum(a[i]>a[j] for i in range(8) for j in range(i+1,8))
    if inv%2: print('This state cannot reach the goal.'); return
    for kind in ('misplaced','manhattan'):
        path,expanded=astar(start,kind)
        print(f'\n{kind.title()} heuristic:')
        print(f'Moves: {len(path)-1}; expanded states: {expanded}')
        for state in path:
            for i in (0,3,6): print(*state[i:i+3])
            print()


SAMPLE_INPUTS = ['1 2 3 4 5 6 7 0 8']
with sample_input(SAMPLE_INPUTS):
    main()


Enter 9 tiles, 0 for blank: 1 2 3 4 5 6 7 0 8

Misplaced heuristic:
Moves: 1; expanded states: 1
1 2 3
4 5 6
7 0 8

1 2 3
4 5 6
7 8 0


Manhattan heuristic:
Moves: 1; expanded states: 1
1 2 3
4 5 6
7 0 8

1 2 3
4 5 6
7 8 0

